# Agri-JAX 教程：基于 JAX 的 DSSAT-CSM 玉米模型

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juksentang/agri-jax/blob/main/tutorial/agrijax_tutorial_zh.ipynb) · [English Version](https://colab.research.google.com/github/juksentang/agri-jax/blob/main/tutorial/agrijax_tutorial.ipynb)

Agri-JAX 基于已发表文献中的数学方程，在 JAX 中重新实现了 DSSAT-CSM v4.8.6 玉米模型（土壤水分、蒸腾蒸发、根系吸水、CERES-Maize），并可直接读取 DSSAT 原生试验文件；我们仅将 DSSAT 原始程序用作基准对齐与校验工具。本项目是基于已发表方程的独立实现，其模拟输出已通过 DSSAT-CSM（BSD-3 协议）和 RZWQM2 的对照验证。DSSAT-CSM 及其数据与示例试验归属于 DSSAT 基金会及各位贡献者（BSD-3 协议）；本教程建立在他们的工作基础之上。

本教程选用的示例为 DSSAT 经典试验 **UFGA8201**（佛罗里达州盖恩斯维尔，1982 年）的处理 4（充分灌溉、高氮）；模型在**关闭氮素模拟**（nitrogen off）状态下运行。

1. **运行 Agri-JAX**：安装依赖、读取试验数据并模拟单个生长季，全程不调用 DSSAT 原始程序。
2. **与 DSSAT 对比**：获取 DSSAT 原生程序（经过验证的 v4.8.6.0 静态编译版本），叠加其模拟结果进行对照。
3. **批处理与速度**：单次批处理调用同时模拟气象年份 × 播种日期 × 品种样本，在当前机器上测速，并以同机 DSSAT 作为基准。
4. **参数校准**：完全在 Agri-JAX 内部基于处理 4 拟合品种遗传参数，处理 6 作为留出集（验证集）。
5. **写回 DSSAT 验证**：将校准后的 `.CUL` 参数行放入 DSSAT 中运行核验。

在免费 CPU 运行时环境下，*代码执行程序 → 全部运行*（Runtime → Run all）仅需数分钟。下方打印的所有耗时均在当前会话中测得，测试机器信息随耗时一并输出。

## 1. 运行 Agri-JAX

在 Colab 上，下一个单元格会从 GitHub 安装带有绘图附加组件（plotting extra）的 Agri-JAX（目前 PyPI 上的安装包仅为占位）；在其他环境中执行该单元格不会进行任何操作（可通过 `pip install "agrijax[plot] @ git+https://github.com/juksentang/agri-jax"` 安装）。若安装后 Colab 提示重启会话，请重启会话并从其后的单元格继续运行。

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q "agrijax[plot] @ git+https://github.com/juksentang/agri-jax"
LANG = "zh"  # 本笔记本输出信息的语言（"en" 或 "zh"）

In [ ]:
# Printed messages of this notebook in English and Chinese (tutorial/source/text.toml)
MSG = {
    "machine": {
        "en": "machine: {machine}",
        "zh": "测试机器：{machine}",
    },
    "season": {
        "en": "Agri-JAX, {treatment}: silking {adat}, maturity {mdat}, grain yield {hwam:.0f} kg/ha",
        "zh": "Agri-JAX，{treatment}：吐丝期 {adat}，成熟期 {mdat}，籽粒产量 {hwam:.0f} kg/ha",
    },
    "scenarios": {
        "en": "{n} scenarios built from DSSAT's files (no DSSAT run)",
        "zh": "基于 DSSAT 文件构建了 {n} 个情景（未运行 DSSAT）",
    },
    "check_yield": {
        "en": "published cultivar on {n} scenarios: Agri-JAX yield within {rel:.3%} of DSSAT",
        "zh": "已发表品种在 {n} 个情景下：Agri-JAX 产量与 DSSAT 的相对误差在 {rel:.3%} 以内",
    },
    "check_dates": {
        "en": "silking date equal on {adat} and maturity date on {mdat} of {n} scenarios",
        "zh": "{n} 个情景中：吐丝期一致数 {adat}，成熟期一致数 {mdat}",
    },
    "dssat_per_scenario": {
        "en": "DSSAT took {sec:.2f} s per scenario on this machine (one process each)",
        "zh": "DSSAT 在当前机器上平均每情景耗时 {sec:.2f} s（每个情景单独立进程）",
    },
    "batch_yield": {
        "en": "{n} cultivars, 1982: Agri-JAX yield within {rel:.3%} of DSSAT",
        "zh": "1982 季，{n} 个品种：Agri-JAX 产量与 DSSAT 的相对误差在 {rel:.3%} 以内",
    },
    "batch_dates": {
        "en": "maturity date equal on {same} of {n}",
        "zh": "{n} 个品种中成熟期一致数：{same}",
    },
    "dssat_speed": {
        "en": "DSSAT:    {ms:.2f} ms per season (measured: {n} seasons in {s:.2f} s, one batch run)",
        "zh": "DSSAT:    每季 {ms:.2f} ms（实测：单次批处理运行 {s:.2f} s 完成 {n} 个生长季）",
    },
    "ours_speed": {
        "en": "Agri-JAX: {ms:.2f} ms per season (measured: {n} seasons in {s:.2f} s)",
        "zh": "Agri-JAX: 每季 {ms:.2f} ms（实测：{s:.2f} s 完成 {n} 个生长季）",
    },
    "compile": {
        "en": "Agri-JAX compiled its batched program once, in {s:.1f} s (first call)",
        "zh": "Agri-JAX 批处理程序单次编译耗时：{s:.1f} s（首次调用）",
    },
    "extrapolated": {
        "en": "{n} seasons: DSSAT about {s:.0f} s (EXTRAPOLATED from its measured per-season time)",
        "zh": "{n} 个生长季：DSSAT 预估耗时约 {s:.0f} s（根据单季实测耗时外推）",
    },
}


def say(key, **values):
    """Print message ``key`` in the language of this notebook."""
    print(MSG[key][LANG].format(**values))

In [ ]:
import os

import numpy as np

import agrijax as aj

SMALL = os.environ.get("AGRI_JAX_TUTORIAL_SMALL") == "1"  # 快速运行，供测试使用
say("machine", machine=aj.machine())

`aj.dssat.experiment("UFGA8201")` 用于读取 DSSAT 示例试验。其文件均来自 DSSAT 公开数据，以固定版本从 DSSAT 官方仓库下载一次，并通过 SHA-256 完整性校验：包括从 `DSSAT/dssat-csm-data` 下载的 FileX、实测数据、盖恩斯维尔 1978–1987 年气象数据及土壤文件，以及从 `DSSAT/dssat-csm-os` v4.8.6.0（BSD-3 协议）下载的玉米品种、生态型、物种参数文件和标准参考数据。

表格列出了各试验处理及其对应品种、Agri-JAX 是否能直接构建该处理的输入，以及在开启氮素模拟时 DSSAT 原生模型的产量变动幅度。由于当前模型在关闭氮素模拟下运行，受氮素影响显著的处理（低氮处理 1、3、5 以及处理 2）不参与拟合；拟合使用的是处理 4 和处理 6。

In [ ]:
exp = aj.dssat.experiment("UFGA8201")
print(exp)
exp.table()

`exp.run(treatment=4)` 仅使用 Agri-JAX 模拟该生长季。输入数据由 Agri-JAX 实现的 DSSAT 输入解析链直接从试验文件中构建：土壤分层及初始含水量提取自 FileX 与 `.SOL`，逐日气象提取自 `.WTH`，并读取灌溉计划，以及 `.CUL` / `.ECO` / `.SPE` 中的品种、生态型和物种参数；模拟从起始日一直推进至作物自身生理成熟。首次运行还会触发模型编译，该编译耗时计入单元格总运行时间。配图展示了叶面积指数、地上部生物量与籽粒干重，以及土壤剖面储水量，并叠加了试验 T 文件（时序实测）中的观测值（圆圈标记）。

In [ ]:
season = exp.run(treatment=4)
s = season.summary
say("season", treatment=season.treatment, adat=int(s["ADAT"]), mdat=int(s["MDAT"]), hwam=s["HWAM"])
aj.plot.season(season, observed=exp.observed(4));

In [ ]:
season.to_frame().tail()

## 2. 与 DSSAT 对比

`aj.dssat.install_reference()` 获取 DSSAT 原生程序 `dscsm048` v4.8.6.0：该预编译静态链接版本用于 Agri-JAX 所有的基准验证运行（作为 Agri-JAX 仓库的 Release 资源提供，经 SHA-256 校验；附带 DSSAT 开源许可及来源说明）。`exp.reference(treatment=4)` 在同一处理上运行该原生程序（关闭氮素模拟）。

生长季末输出值：吐丝期（`ADAT`）与成熟期（`MDAT`）日期，以及相差天数；成熟期籽粒产量（`HWAM`）与成熟期地上部总重（`CWAM`），单位为 kg ha⁻¹，以及相对差异。

In [ ]:
aj.dssat.install_reference()
ref = exp.reference(treatment=4)
season.compare_summary(ref)

逐日对比：在两者均有输出的每一天，比对叶面积指数、地上部总重与籽粒干重（`PlantGro.OUT`）以及土壤剖面储水量（`SoilWat.OUT`）。DSSAT 输出的这些数值经过了舍入截断（LAI 保留至 0.01，生物量保留至 1 kg ha⁻¹，土壤储水量保留至 1 mm），这决定了两者可见数值差异的理论下限。配图叠加了 DSSAT 的运行曲线（虚线），并绘制了 Agri-JAX 与 DSSAT 的散点对照图。

In [ ]:
season.compare_daily(ref)

In [ ]:
aj.plot.season(season, reference=ref, observed=exp.observed(4));

In [ ]:
aj.plot.compare(season, ref);

## 3. 批处理与速度

将同一处理推广至其他气象年份与播种日期：`exp.scenarios` 将 FileX 中的整个农事管理日历（模拟起始、播种、灌溉、施肥日期）映射平移到各个气象年份并施加不同的播期偏移量，直接构建各情景的输入数据，全程无需运行 DSSAT。这些情景不属于原始验证集，因此在下文中会逐一与 DSSAT 进行对照核验。这仅是单站点下单品种在关闭氮素模拟下的批量推演：旨在演示工作流程，而非实际农艺播种期推荐。

In [ ]:
years = range(1981, 1984) if SMALL else range(1978, 1988)
shifts = [0] if SMALL else [-14, 0, 14]
scen = exp.scenarios(treatment=4, years=years, sowing_shift=shifts)
say("scenarios", n=len(scen.runs))
scen.table

`scen.run(cultivar)` 在单次批处理程序中同时模拟所有情景下的所有品种样本：底层在时间维度上用 `scan` 沿天数迭代，外层在生长季维度上用 `vmap` 向量化，仅需编译一次。其计时逻辑（`timing`）将编译耗时（`compile_s`，若此前已编译则为 0）与纯运行耗时（`run_s`，包含将结果传回主机内存）区分开来。下方的第二次调用复用了已编译程序并传入新样本：该耗时代表了稳态开销（steady-state cost）。配图展示了各气象年份下所有样本的产量分布。

In [ ]:
K = 8 if SMALL else 320
pub = scen.published


def samples(seed):
    """在已发表品种参数的 ±10% 范围内采样的 K 个品种；样本 0 为已发表的基准品种。"""
    rng = np.random.default_rng(seed)
    c = {n: pub[n] * rng.uniform(0.9, 1.1, K) for n in ("P1", "P5", "G2", "G3", "PHINT")}
    for n in c:
        c[n][0] = pub[n]
    return c


first = scen.run(samples(0))
first

In [ ]:
second = scen.run(samples(1))
second

In [ ]:
aj.plot.batch(first, by="year");

`scen.reference()` 使用已发表的基准品种在每个情景上运行 DSSAT（每个情景单独立进程）。每个情景中的样本 0 均为该基准品种，因此 Agri-JAX 与 DSSAT 在每个情景下的输出必须完全一致。

In [ ]:
dss = scen.reference()
pub_runs = first.table[first.table["sample"] == 0]
t = pub_runs.merge(dss, on=["year", "sowing_shift"], suffixes=("", " DSSAT"))
rel = (t["HWAM"] - t["HWAM DSSAT"]).abs() / t["HWAM DSSAT"]
same_adat = int((t["ADAT"] == t["ADAT DSSAT"]).sum())
same_mdat = int((t["MDAT"] == t["MDAT DSSAT"]).sum())
say("check_yield", n=len(t), rel=rel.max())
say("check_dates", adat=same_adat, mdat=same_mdat, n=len(t))
say("dssat_per_scenario", sec=float(np.mean(scen.dssat_s)))

**在当前机器上与 DSSAT 同机测速对比。** `exp.dssat_batch` 在单次 DSSAT 批处理运行中（单进程；耗时包含了准备运行目录及落盘写入逐日输出文件的时间），对 1982 生长季的一组扰动品种参数运行 DSSAT。Agri-JAX 对写入的同一组参数进行模拟；两者的数值必须保持一致。随后打印两者折算到单生长季的平均耗时。整个批次的 DSSAT 总耗时是由测得的单季耗时**外推估算**得出的，并已据此标明；此外，两个程序在每个生长季中所完成的工作量也有所不同（DSSAT 需要写入完整的逐日输出文件，并运行了 Agri-JAX 单日步长核心略去的模块，而 Agri-JAX 每个生长季仅返回 4 个标量值）。

In [ ]:
n_dssat = 5 if SMALL else 50
db = exp.dssat_batch(treatment=4, cultivar={n: v[:n_dssat] for n, v in samples(2).items()})
ours = scen.run(db.cultivar).table
ours = ours[(ours["year"] == 1982) & (ours["sowing_shift"] == 0)].reset_index(drop=True)
rel = (ours["HWAM"] - db.table["HWAM"]).abs() / db.table["HWAM"]
say("batch_yield", n=db.seasons, rel=rel.max())
say("batch_dates", same=int((ours["MDAT"] == db.table["MDAT"]).sum()), n=db.seasons)

In [ ]:
n = second.timing["seasons"]
run_s = second.timing["run_s"]
per_dssat = db.elapsed_s / db.seasons
say("machine", machine=aj.machine())
say("dssat_speed", ms=1e3 * per_dssat, n=db.seasons, s=db.elapsed_s)
say("ours_speed", ms=1e3 * run_s / n, n=n, s=run_s)
say("compile", s=first.timing["compile_s"])
say("extrapolated", n=n, s=n * per_dssat)

## 4. 参数校准

`aj.calibrate` 在参数文件规定的 MINIMA / MAXIMA 范围内，通过多起点联合 CMA-ES 算法，完全在 Agri-JAX 内部将品种的 6 个 `MZCER048.CUL` 遗传参数（P1、P2、P5、G2、G3、PHINT）拟合至处理 4 的观测数据（吐丝期与成熟期、收获期产量、地上部总重、收获期粒数以及逐日 LAI 时序）；若观测数据未包含某参数的有效约束信息，则该参数固定为文献公开发表值并在日志中报告。处理 6 作为留出集（held out）：仅用于泛化评估，不参与拟合。为了适配免费运行时环境，此处的计算预算设置较小（8 个起点 × 200 次评估；快速运行模式下评估次数更少）；最终结果在 `.CUL` 数据行所能容纳的数值精度内进行筛选，若没有优于基准的结果，则保留已发表的基准品种。

In [ ]:
starts, budget = (2, 48) if SMALL else (8, 200)
res = aj.calibrate(exp, treatments=[4], holdout=[6], starts=starts, budget=budget, seed=0)
print(res)

In [ ]:
aj.plot.calibration(res);

## 5. 写回 DSSAT 验证

将校准后的参数行写入 DSSAT `MZCER048.CUL` 的副本中，随后 `check_dssat()` 使用该参数行在两个处理上分别运行原生 DSSAT 程序：DSSAT 模拟的吐丝期与成熟期必须与 Agri-JAX 完全一致，且产量相对误差必须在 0.1% 以内。

In [ ]:
if os.path.exists("MZCER048_calibrated.CUL"):
    os.remove("MZCER048_calibrated.CUL")
print(res.write_cul("MZCER048_calibrated.CUL"))
res.check_dssat()

## 后续探索

- `exp.run` 与 `exp.scenarios` 同样支持其他 DSSAT 玉米试验（`aj.dssat.experiment("path/to/EXP.MZX")`，对应气象文件位于数据目录的 `example_data/Weather` 中）；若处理所依赖的 DSSAT 物理过程 Agri-JAX 尚未实现（如自动灌溉、带地表残茬的 CENTURY 有机质模型、耕作、地下水位等），程序会明确拒绝并说明原因。`aj.calibrate` 目前仅支持测定了氮素效应的处理（即 DSSAT v4.8.6 玉米示例处理）。由于目前仅对 DSSAT 玉米示例处理完成了验证，应用于其他试验时应参照本笔记本的方法与 DSSAT 进行对照核验。
- 模型在关闭氮素模拟下运行：请选用氮素不对作物生长构成限制的处理（参考 `exp.table()`）。
- 源码、验证与过程实现规则：[github.com/juksentang/agri-jax](https://github.com/juksentang/agri-jax)。DSSAT-CSM 版权所有 © DSSAT 基金会及其贡献者（BSD-3 协议）；示例数据来源于 `DSSAT/dssat-csm-data` 与 `DSSAT/dssat-csm-os`。